## 0. Kiem tra moi truong va kernel

Cell nay kiem tra Python interpreter, kernel Jupyter, virtualenv/conda, va xac nhan ban dang mo notebook bang moi truong phu hop.


In [1]:
# Thiết lập thư mục làm việc local và chẩn đoán môi trường trong VS Code / Jupyter
from pathlib import Path
import os
import platform
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
WORK_DIR = PROJECT_ROOT / 'workspace'
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

EXPECTED_KERNEL_NAME = 'tf-metal'
EXPECTED_CONDA_ENV = 'tf-metal'
venv_path = os.environ.get('VIRTUAL_ENV')
conda_prefix = os.environ.get('CONDA_PREFIX')
conda_env_name = os.environ.get('CONDA_DEFAULT_ENV')
IS_MACOS = platform.system() == 'Darwin'
IS_APPLE_SILICON = IS_MACOS and platform.machine() == 'arm64'
ipykernel_launcher = 'ipykernel_launcher' in ' '.join(sys.argv)
kernel_connection_file = Path(os.environ.get('JPY_CONNECTION_FILE', '')).name if os.environ.get('JPY_CONNECTION_FILE') else 'Không rõ'
looks_like_system_python = sys.prefix == getattr(sys, 'base_prefix', sys.prefix) and not venv_path and not conda_prefix
using_expected_tf_metal_env = EXPECTED_CONDA_ENV in sys.executable or conda_env_name == EXPECTED_CONDA_ENV or (conda_prefix and conda_prefix.endswith(f'/{EXPECTED_CONDA_ENV}'))

apple_gpu_summary = 'Không xác định'
if IS_MACOS:
    try:
        sp_output = subprocess.check_output(['system_profiler', 'SPDisplaysDataType'], text=True)
        if 'Apple M' in sp_output or 'Metal: Supported' in sp_output:
            apple_gpu_summary = 'Phát hiện Apple GPU với Metal support'
        else:
            apple_gpu_summary = 'Không thấy thông tin Apple GPU rõ ràng'
    except Exception:
        apple_gpu_summary = 'Không đọc được system_profiler'

print('=== Môi trường chạy notebook ===')
print(f'Python executable: {sys.executable}')
print(f'Python version: {platform.python_version()}')
print(f'sys.prefix: {sys.prefix}')
print(f'Đang chạy qua ipykernel: {ipykernel_launcher}')
print(f'Kernel connection file: {kernel_connection_file}')
print(f'VIRTUAL_ENV: {venv_path or "Không có"}')
print(f'CONDA_PREFIX: {conda_prefix or "Không có"}')
print(f'CONDA_DEFAULT_ENV: {conda_env_name or "Không có"}')
print(f'Kernel mong muốn: {EXPECTED_KERNEL_NAME}')
print(f'Apple Silicon: {IS_APPLE_SILICON}')
print(f'Trạng thái GPU trên macOS: {apple_gpu_summary}')

if conda_prefix or venv_path:
    print('Kernel hiện tại có vẻ đang dùng môi trường ảo, phù hợp để chạy notebook.')
elif looks_like_system_python:
    print('Cảnh báo: notebook có vẻ đang chạy bằng Python hệ thống.')
    print('Nếu bạn đã tạo venv/conda riêng cho dự án, hãy đổi kernel trong VS Code để tránh lỗi package.')
else:
    print('Không phát hiện venv/conda rõ ràng, nhưng kernel hiện tại vẫn có thể dùng được nếu package đầy đủ.')

if IS_APPLE_SILICON and not using_expected_tf_metal_env:
    print('CANH BAO: Ban dang mo notebook bang kernel khac tf-metal, nen TensorFlow co the se chay CPU.')
    print('Trong VS Code, hay chon kernel: Python 3.10 (tf-metal).')
elif IS_APPLE_SILICON and using_expected_tf_metal_env:
    print('Dang dung dung kernel tf-metal. Neu moi truong TensorFlow on dinh, notebook co the dung GPU.')

print(f'Thư mục project: {PROJECT_ROOT}')
print(f'Thư mục làm việc: {WORK_DIR}')

=== Môi trường chạy notebook ===
Python executable: /opt/anaconda3/envs/tf-metal/bin/python
Python version: 3.10.20
sys.prefix: /opt/anaconda3/envs/tf-metal
Đang chạy qua ipykernel: True
Kernel connection file: Không rõ
VIRTUAL_ENV: Không có
CONDA_PREFIX: /opt/anaconda3/envs/tf-metal
CONDA_DEFAULT_ENV: tf-metal
Kernel mong muốn: tf-metal
Apple Silicon: True
Trạng thái GPU trên macOS: Phát hiện Apple GPU với Metal support
Kernel hiện tại có vẻ đang dùng môi trường ảo, phù hợp để chạy notebook.
Dang dung dung kernel tf-metal. Neu moi truong TensorFlow on dinh, notebook co the dung GPU.
Thư mục project: /Applications/Python_AI/Neural_Image_Caption_Generation
Thư mục làm việc: /Applications/Python_AI/Neural_Image_Caption_Generation/workspace


In [2]:
# Kiểm tra và cài package theo đúng interpreter mà VS Code đang dùng
import importlib
import importlib.util
import shutil
import subprocess
import sys

REQUIRED_PACKAGES = {
    'ipykernel': 'ipykernel',
    'numpy': 'numpy',
    'scipy': 'scipy',
    'nltk': 'nltk',
    'tensorflow': 'tensorflow',
    'PIL': 'pillow',
}

OPTIONAL_PACKAGES = {}
if IS_APPLE_SILICON:
    OPTIONAL_PACKAGES['tensorflow_metal'] = 'tensorflow-metal'

missing_packages = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    print('Đang cài các package còn thiếu:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
    print('Cài đặt hoàn tất.')
    print('Vui lòng Restart Kernel trong VS Code rồi chạy lại từ cell đầu để Python nạp package mới đúng cách.')
else:
    print('Môi trường đã đủ package cần thiết.')

missing_optional_packages = []
for module_name, package_name in OPTIONAL_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        missing_optional_packages.append(package_name)

if missing_optional_packages:
    print('Thiếu package tăng tốc tùy chọn:', missing_optional_packages)
    print('Trên Mac Apple Silicon, tensorflow-metal là hướng dùng GPU qua Metal, nhưng cần kiểm tra tương thích phiên bản trước khi cài.')
else:
    if OPTIONAL_PACKAGES:
        print('Các package tăng tốc tùy chọn đã sẵn sàng.')

# Kiểm tra GPU nếu máy có cài NVIDIA utilities
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('Không tìm thấy nvidia-smi. Nếu bạn chạy CPU thì có thể bỏ qua.')

print(f'Python executable (cell cài package): {sys.executable}')

Môi trường đã đủ package cần thiết.
Thiếu package tăng tốc tùy chọn: ['tensorflow-metal']
Trên Mac Apple Silicon, tensorflow-metal là hướng dùng GPU qua Metal, nhưng cần kiểm tra tương thích phiên bản trước khi cài.
Không tìm thấy nvidia-smi. Nếu bạn chạy CPU thì có thể bỏ qua.
Python executable (cell cài package): /opt/anaconda3/envs/tf-metal/bin/python


In [3]:
import os
import string
import re
import gc
import json
import math
import importlib.util
import numpy as np
import zipfile
from PIL import Image
from urllib.request import urlretrieve
from pickle import dump, load
from os import listdir
from os.path import join

import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, Add
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from nltk.translate.bleu_score import corpus_bleu
import nltk
nltk.download('punkt', quiet=True)

gpu_devices = tf.config.list_physical_devices('GPU')
has_tensorflow_metal = importlib.util.find_spec('tensorflow_metal') is not None

print(f'Pillow version: {Image.__version__}')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU khả dụng: {gpu_devices}')
print(f'Thư mục hiện tại: {Path.cwd()}')
print(f'WORK_DIR tồn tại: {WORK_DIR.exists()}')
print(f'tensorflow-metal đã cài: {has_tensorflow_metal}')

if IS_APPLE_SILICON and not gpu_devices:
    if has_tensorflow_metal:
        print('Mac có Apple GPU, nhưng TensorFlow hiện vẫn đang chạy CPU. Hãy kiểm tra phiên bản tensorflow/tensorflow-metal và kernel VS Code.')
    else:
        print('Mac có Apple GPU, nhưng TensorFlow hiện đang chạy CPU. Môi trường này chưa có tensorflow-metal, hoặc chưa có cấu hình tương thích để dùng Metal.')
elif gpu_devices:
    print('TensorFlow đã nhận GPU để tăng tốc.')
else:
    print('TensorFlow hiện đang chạy CPU.')

Pillow version: 12.1.1
TensorFlow version: 2.16.2
GPU khả dụng: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Thư mục hiện tại: /Applications/Python_AI/Neural_Image_Caption_Generation/workspace
WORK_DIR tồn tại: True
tensorflow-metal đã cài: False
TensorFlow đã nhận GPU để tăng tốc.


## 1. Tai dataset Flickr8k

Dataset được host trên GitHub releases của tác giả Jason Brownlee. Gồm 2 file zip:
- `Flickr8k_Dataset.zip` (~1GB): 8,092 ảnh JPEG
- `Flickr8k_text.zip` (~3MB): file mô tả, file phân chia train/dev/test

In [4]:
# Tải dataset Flickr8k từ GitHub releases
flickr8k_images = os.path.join(WORK_DIR, 'Flicker8k_Dataset')
flickr8k_text = os.path.join(WORK_DIR, 'Flickr8k_text')
if not os.path.exists(flickr8k_text):
    flickr8k_text = str(WORK_DIR)

def download_and_extract(url, archive_name, output_dir):
    archive_path = os.path.join(WORK_DIR, archive_name)
    print(f'Đang tải {archive_name}...')
    urlretrieve(url, archive_path)
    with zipfile.ZipFile(archive_path, 'r') as z:
        z.extractall(output_dir)
    os.remove(archive_path)
    print(f'Đã giải nén {archive_name}.')

if not os.path.exists(flickr8k_images):
    download_and_extract(
        'https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip',
        'Flickr8k_Dataset.zip',
        WORK_DIR,
    )
    print(f'Giải nén xong. Số ảnh: {len(listdir(flickr8k_images))}')
else:
    print(f'Dataset ảnh đã có. Số ảnh: {len(listdir(flickr8k_images))}')

if not os.path.exists(os.path.join(flickr8k_text, 'Flickr8k.token.txt')):
    download_and_extract(
        'https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip',
        'Flickr8k_text.zip',
        WORK_DIR,
    )
    if os.path.exists(os.path.join(WORK_DIR, 'Flickr8k_text')):
        flickr8k_text = os.path.join(WORK_DIR, 'Flickr8k_text')
    else:
        flickr8k_text = str(WORK_DIR)
else:
    print('Dataset text đã có.')

Dataset ảnh đã có. Số ảnh: 8091
Dataset text đã có.


## 2. Trich xuat dac trung anh VGG16

Cell nay trich xuat hai loai dac trung anh tu VGG16: `features_fc` va `features_conv`, sau do luu ra file pickle de tai su dung.


In [5]:
def extract_features_dual(directory):
    """Trích xuất đặc trưng ảnh từ VGG16.

    Trả về hai dict:
    - features_fc: {image_id: np.array (1, 4096)} cho mô hình LSTM
    - features_conv: {image_id: np.array (49, 512)} cho mô hình Transformer

    Lưu ý shape:
    - features_fc lưu shape (1, 4096) — GIỮ batch dimension vì model_fc.predict()
      trả về (1, 4096). Khi tạo training data, photos[key][0] lấy ra vector (4096,)
      rồi append vào list, cuối cùng np.array() stack thành (N, 4096).
    - features_conv lưu shape (49, 512) — ĐÃ BỎ batch dimension. Khi tạo training data,
      photos[key] trực tiếp append, np.array() stack thành (N, 49, 512).
    """
    # Tải mô hình VGG16 đã huấn luyện trên ImageNet
    base_model = VGG16()

    # Mô hình 1: đầu ra lớp FC thứ hai từ cuối → (1, 4096)
    model_fc = Model(inputs=base_model.inputs,
                     outputs=base_model.layers[-2].output)

    # Mô hình 2: đầu ra block5_pool → (1, 7, 7, 512)
    model_conv = Model(inputs=base_model.inputs,
                       outputs=base_model.get_layer('block5_pool').output)

    features_fc = dict()
    features_conv = dict()
    filenames = listdir(directory)
    total = len(filenames)

    for i, name in enumerate(filenames):
        filepath = join(directory, name)
        # Tải ảnh, resize về 224×224, chuyển thành mảng numpy
        image = load_img(filepath, target_size=(224, 224))
        image = img_to_array(image)
        # Thêm batch dimension: (224,224,3) → (1,224,224,3)
        image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
        # Tiền xử lý theo chuẩn ImageNet (trừ mean RGB)
        image = preprocess_input(image)

        # Trích xuất đặc trưng
        feat_fc = model_fc.predict(image, verbose=0)      # (1, 4096)
        feat_conv = model_conv.predict(image, verbose=0)   # (1, 7, 7, 512)

        # Lấy image_id từ tên file (bỏ phần mở rộng .jpg)
        image_id = name.split('.')[0]
        features_fc[image_id] = feat_fc                     # giữ (1, 4096)
        features_conv[image_id] = feat_conv.reshape(
            feat_conv.shape[1] * feat_conv.shape[2], feat_conv.shape[3]
        )  # (7*7, 512) = (49, 512)

        if (i + 1) % 500 == 0:
            print(f'  Đã xử lý {i+1}/{total} ảnh...')

    return features_fc, features_conv

# Kiểm tra file đã trích xuất chưa — tránh chạy lại
fc_path = os.path.join(WORK_DIR, 'features_fc.pkl')
conv_path = os.path.join(WORK_DIR, 'features_conv.pkl')

if not (os.path.exists(fc_path) and os.path.exists(conv_path)):
    print('Trích xuất đặc trưng ảnh (mất ~10-30 phút)...')
    features_fc, features_conv = extract_features_dual(flickr8k_images)
    with open(fc_path, 'wb') as fc_file:
        dump(features_fc, fc_file)
    with open(conv_path, 'wb') as conv_file:
        dump(features_conv, conv_file)
    print(f'Đã lưu. Số ảnh: {len(features_fc)}')
    print(f'  features_fc shape mẫu: {list(features_fc.values())[0].shape}')
    print(f'  features_conv shape mẫu: {list(features_conv.values())[0].shape}')
else:
    print('Đặc trưng ảnh đã có sẵn, bỏ qua bước trích xuất.')

Đặc trưng ảnh đã có sẵn, bỏ qua bước trích xuất.


## 3. Lam sach mo ta caption

Cell nay doc caption goc, lam sach text, tao `descriptions.txt`, va chuan bi du lieu mo ta cho cac buoc train/test sau do.


In [17]:
def load_doc(filename):
    """Đọc toàn bộ nội dung file văn bản."""
    with open(filename, 'r') as f:
        return f.read()

def load_descriptions(doc):
    """Phân tích file Flickr8k.token.txt thành dict {image_id: [desc1, desc2, ...]}.

    Mỗi ảnh có 5 mô tả. image_id là tên file bỏ phần mở rộng .jpg.
    Format dòng: tên_file.jpg#N<tab>mô_tả
    """
    mapping = dict()
    for line in doc.split('\n'):
        tokens = line.split()
        if len(line) < 2:
            continue
        # tokens[0] = 'xxx.jpg#0', tokens[1:] = các từ trong mô tả
        image_id, image_desc = tokens[0], tokens[1:]
        # Bỏ phần .jpg#N, chỉ giữ image_id
        image_id = image_id.split('.')[0]
        image_desc = ' '.join(image_desc)
        if image_id not in mapping:
            mapping[image_id] = list()
        mapping[image_id].append(image_desc)
    return mapping

def clean_descriptions(descriptions):
    """Làm sạch tất cả mô tả: viết thường, bỏ dấu câu, bỏ từ ngắn/chứa số."""
    re_punc = re.compile('[%s]' % re.escape(string.punctuation))
    for _, desc_list in descriptions.items():
        for i in range(len(desc_list)):
            desc = desc_list[i]
            desc = desc.split()
            # Chuyển chữ thường
            desc = [word.lower() for word in desc]
            # Bỏ dấu câu
            desc = [re_punc.sub('', w) for w in desc]
            # Bỏ từ có 1 ký tự trở xuống (sau khi bỏ dấu câu)
            desc = [word for word in desc if len(word) > 1]
            # Chỉ giữ từ toàn chữ cái (bỏ từ chứa số)
            desc = [word for word in desc if word.isalpha()]
            desc_list[i] = ' '.join(desc)

def to_vocabulary(descriptions):
    """Tạo tập từ vựng (set) từ tất cả mô tả."""
    all_desc = set()
    for key in descriptions.keys():
        [all_desc.update(d.split()) for d in descriptions[key]]
    return all_desc

def save_descriptions(descriptions, filename):
    """Lưu mô tả đã làm sạch ra file text.
    Mỗi dòng: image_id mô_tả (không có startseq/endseq ở bước này).
    """
    lines = list()
    for key, desc_list in descriptions.items():
        for desc in desc_list:
            lines.append(key + ' ' + desc)
    data = '\n'.join(lines)
    with open(filename, 'w') as f:
        f.write(data)

# --- Thực thi pipeline làm sạch ---
token_file = os.path.join(flickr8k_text, 'Flickr8k.token.txt')
doc = load_doc(token_file)
descriptions = load_descriptions(doc)
print(f'Số ảnh có mô tả: {len(descriptions)}')

# In mẫu trước khi làm sạch
sample_key = list(descriptions.keys())[0]
print(f'\nMẫu trước làm sạch ({sample_key}):')
for d in descriptions[sample_key][:2]:
    print(f'  {d}')

clean_descriptions(descriptions)
vocabulary = to_vocabulary(descriptions)
print(f'\nKích thước từ vựng sau làm sạch: {len(vocabulary)}')

# In mẫu sau khi làm sạch
print(f'\nMẫu sau làm sạch ({sample_key}):')
for d in descriptions[sample_key][:2]:
    print(f'  {d}')

# Lưu file
desc_file = os.path.join(WORK_DIR, 'descriptions.txt')
save_descriptions(descriptions, desc_file)
print(f'\nĐã lưu mô tả vào: {desc_file}')

Số ảnh có mô tả: 8092

Mẫu trước làm sạch (1000268201_693b08cb0e):
  A child in a pink dress is climbing up a set of stairs in an entry way .
  A girl going into a wooden building .

Kích thước từ vựng sau làm sạch: 8763

Mẫu sau làm sạch (1000268201_693b08cb0e):
  child in pink dress is climbing up set of stairs in an entry way
  girl going into wooden building

Đã lưu mô tả vào: /Applications/Python_AI/Neural_Image_Caption_Generation/workspace/descriptions.txt


## 4. Nap train/dev/test va tokenizer

Cell nay nap train/dev/test split, descriptions da clean, feature files, va tao tokenizer cho tap train.


In [18]:
def load_set(filename):
    """Tải danh sách image_id từ file phân chia train/dev/test.

    File có format: mỗi dòng là tên_file.jpg
    """
    doc = load_doc(filename)
    dataset = list()
    for line in doc.split('\n'):
        if len(line) < 1:
            continue
        identifier = line.split('.')[0]
        dataset.append(identifier)
    return set(dataset)

def load_clean_descriptions(filename, dataset):
    """Tải mô tả đã làm sạch, thêm startseq/endseq, lọc theo dataset."""
    doc = load_doc(filename)
    descriptions = dict()
    for line in doc.split('\n'):
        tokens = line.split()
        if len(tokens) < 2:
            continue
        image_id, image_desc = tokens[0], tokens[1:]
        if image_id in dataset:
            if image_id not in descriptions:
                descriptions[image_id] = list()
            desc = 'startseq ' + ' '.join(image_desc) + ' endseq'
            descriptions[image_id].append(desc)
    return descriptions

def load_photo_features(filename, dataset):
    """Tải đặc trưng ảnh từ file pickle, chỉ giữ ảnh trong dataset."""
    with open(filename, 'rb') as feature_file:
        all_features = load(feature_file)
    return {k: all_features[k] for k in dataset if k in all_features}

def to_lines(descriptions):
    all_desc = list()
    for key in descriptions.keys():
        [all_desc.append(d) for d in descriptions[key]]
    return all_desc

def create_tokenizer(descriptions, max_vocab_size=7000):
    """Tạo tokenizer có giới hạn vocab vừa phải để giảm unk nhưng vẫn ổn định khi train."""
    lines = to_lines(descriptions)
    tokenizer = Tokenizer(num_words=max_vocab_size, oov_token='unk')
    tokenizer.fit_on_texts(lines)
    return tokenizer

def calc_max_length(descriptions):
    lines = to_lines(descriptions)
    return max(len(d.split()) for d in lines)

print('--- Tải dữ liệu train ---')
train_file = os.path.join(flickr8k_text, 'Flickr_8k.trainImages.txt')
train_set = load_set(train_file)
print(f'Số ảnh train: {len(train_set)}')

train_descriptions = load_clean_descriptions(desc_file, train_set)
print(f'Số ảnh có mô tả trong train: {len(train_descriptions)}')

train_features_fc = load_photo_features(fc_path, train_set)
print(f'Đặc trưng FC train: {len(train_features_fc)}')

train_features_conv = load_photo_features(conv_path, train_set)
print(f'Đặc trưng Conv train: {len(train_features_conv)}')

max_vocab_size = 7000
tokenizer = create_tokenizer(train_descriptions, max_vocab_size=max_vocab_size)
raw_vocab_size = len(tokenizer.word_index) + 1
vocab_size = min(max_vocab_size, raw_vocab_size)
print(f'Kích thước từ vựng gốc: {raw_vocab_size}')
print(f'Kích thước từ vựng dùng để train attention: {vocab_size}')

max_length = calc_max_length(train_descriptions)
print(f'Độ dài mô tả tối đa: {max_length}')

sample_desc = to_lines(train_descriptions)[0]
sample_seq = tokenizer.texts_to_sequences([sample_desc])[0]
print(f'\nMẫu tokenization:')
print(f'  Text: {sample_desc}')
print(f'  Sequence: {sample_seq}')

with open(os.path.join(WORK_DIR, 'tokenizer.pkl'), 'wb') as tokenizer_file:
    dump(tokenizer, tokenizer_file)
print('\nĐã lưu tokenizer.')

print('\n--- Tải dữ liệu dev ---')
dev_file = os.path.join(flickr8k_text, 'Flickr_8k.devImages.txt')
dev_set = load_set(dev_file)
dev_descriptions = load_clean_descriptions(desc_file, dev_set)
dev_features_fc = load_photo_features(fc_path, dev_set)
dev_features_conv = load_photo_features(conv_path, dev_set)
print(f'Số ảnh dev: {len(dev_set)}, mô tả: {len(dev_descriptions)}')

print('\n--- Tải dữ liệu test ---')
test_file = os.path.join(flickr8k_text, 'Flickr_8k.testImages.txt')
test_set = load_set(test_file)
test_descriptions = load_clean_descriptions(desc_file, test_set)
test_features_fc = load_photo_features(fc_path, test_set)
test_features_conv = load_photo_features(conv_path, test_set)
print(f'Số ảnh test: {len(test_set)}, mô tả: {len(test_descriptions)}')


--- Tải dữ liệu train ---
Số ảnh train: 6000
Số ảnh có mô tả trong train: 6000
Đặc trưng FC train: 6000
Đặc trưng Conv train: 6000
Kích thước từ vựng gốc: 7580
Kích thước từ vựng dùng để train attention: 7000
Độ dài mô tả tối đa: 34

Mẫu tokenization:
  Text: startseq child in pink dress is climbing up set of stairs in an entry way endseq
  Sequence: [2, 43, 4, 88, 170, 7, 118, 56, 394, 12, 395, 4, 28, 4475, 641, 3]

Đã lưu tokenizer.

--- Tải dữ liệu dev ---
Số ảnh dev: 1000, mô tả: 1000

--- Tải dữ liệu test ---
Số ảnh test: 1000, mô tả: 1000


## 5. Chuan bi GloVe embeddings

Cell nay tim file GloVe o local, tao `embedding_matrix`, va se dung no de khoi tao word embedding cho attention model.
Notebook uu tien `glove.6B.100d.txt` de giu bo nho va thoi gian train o muc hop ly.


In [ ]:
def find_glove_file():
    """Tìm file GloVe local trong một số vị trí phổ biến."""
    candidate_paths = [
        PROJECT_ROOT / 'glove.6B.100d.txt',
        PROJECT_ROOT / 'glove.6B' / 'glove.6B.100d.txt',
        WORK_DIR / 'glove.6B.100d.txt',
        WORK_DIR / 'glove' / 'glove.6B.100d.txt',
    ]
    for path in candidate_paths:
        if path.exists():
            return path
    return None


def build_glove_embedding_matrix(tokenizer, vocab_size, glove_path):
    """Tạo embedding matrix từ file GloVe cho vocab hiện tại."""
    embedding_index = {}
    with open(glove_path, 'r', encoding='utf-8') as glove_file:
        for line in glove_file:
            values = line.strip().split()
            if not values:
                continue
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embedding_index[word] = vector

    embedding_dim = len(next(iter(embedding_index.values())))
    embedding_matrix = np.random.normal(loc=0.0, scale=0.6, size=(vocab_size, embedding_dim)).astype('float32')
    embedding_matrix[0] = 0.0

    hits = 0
    for word, index in tokenizer.word_index.items():
        if index >= vocab_size:
            continue
        vector = embedding_index.get(word)
        if vector is not None:
            embedding_matrix[index] = vector
            hits += 1

    return embedding_matrix, embedding_dim, hits, len(embedding_index)


glove_path = find_glove_file()
embedding_matrix = None
embedding_dim = 100
embedding_trainable = False

if glove_path is None:
    print('Khong tim thay file GloVe local.')
    print('Hay dat file glove.6B.100d.txt vao mot trong cac vi tri sau:')
    print(f'  - {PROJECT_ROOT / "glove.6B.100d.txt"}')
    print(f'  - {PROJECT_ROOT / "glove.6B" / "glove.6B.100d.txt"}')
    print(f'  - {WORK_DIR / "glove.6B.100d.txt"}')
    print(f'  - {WORK_DIR / "glove" / "glove.6B.100d.txt"}')
    print('Notebook van co the train tiep bang embedding ngau nhien neu ban muon.')
else:
    embedding_matrix, embedding_dim, glove_hits, glove_vocab_size = build_glove_embedding_matrix(
        tokenizer=tokenizer,
        vocab_size=vocab_size,
        glove_path=glove_path,
    )
    glove_coverage = glove_hits / max(1, vocab_size - 1)
    print(f'Da tim thay GloVe: {glove_path}')
    print(f'Embedding dim: {embedding_dim}')
    print(f'So tu GloVe load duoc: {glove_hits}/{max(1, vocab_size - 1)} ({glove_coverage:.2%})')
    print(f'Kich thuoc tu dien GloVe: {glove_vocab_size:,}')
    print(f'Embedding trainable: {embedding_trainable}')


## 6. Tao dataset attention

Cell nay tao pipeline dataset streaming cho attention model tu `features_conv`, de train khong ton qua nhieu RAM.


In [19]:
def normalize_photo_feature(photo_feature):
    """Chuẩn hóa đặc trưng ảnh cho cả vector FC và lưới Conv."""
    photo_feature = np.asarray(photo_feature, dtype='float32')
    if photo_feature.ndim == 1:
        norm = np.linalg.norm(photo_feature)
        if norm > 0:
            photo_feature = photo_feature / norm
    elif photo_feature.ndim == 2:
        norms = np.linalg.norm(photo_feature, axis=1, keepdims=True)
        photo_feature = photo_feature / np.maximum(norms, 1e-8)
    return photo_feature

def count_caption_samples(descriptions, photos):
    total = 0
    for key, desc_list in descriptions.items():
        if key not in photos:
            continue
        for desc in desc_list:
            seq_len = len(desc.split())
            total += max(0, seq_len - 1)
    return total

def caption_sample_generator(tokenizer, max_length, descriptions, photos):
    for key, desc_list in descriptions.items():
        if key not in photos:
            continue
        photo_feature = normalize_photo_feature(photos[key])
        for desc in desc_list:
            seq = tokenizer.texts_to_sequences([desc])[0]
            for i in range(1, len(seq)):
                in_seq, out_seq = seq[:i], seq[i]
                in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0].astype('int32')
                yield (photo_feature.astype('float32'), in_seq), np.int32(out_seq)

def build_caption_dataset(tokenizer, max_length, descriptions, photos, batch_size, shuffle=False):
    output_signature = ((
        tf.TensorSpec(shape=(49, 512), dtype=tf.float32),
        tf.TensorSpec(shape=(max_length,), dtype=tf.int32),
    ), tf.TensorSpec(shape=(), dtype=tf.int32))
    dataset = tf.data.Dataset.from_generator(
        lambda: caption_sample_generator(tokenizer, max_length, descriptions, photos),
        output_signature=output_signature,
    )
    if shuffle:
        sample_count = count_caption_samples(descriptions, photos)
        dataset = dataset.shuffle(min(sample_count, 4096), reshuffle_each_iteration=True)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def clear_legacy_training_arrays():
    legacy_names = ['X1train', 'X2train', 'ytrain', 'X1dev', 'X2dev', 'ydev']
    removed = []
    for name in legacy_names:
        if name in globals():
            del globals()[name]
            removed.append(name)
    gc.collect()
    if removed:
        print(f'Da giai phong bien RAM cu: {removed}')

clear_legacy_training_arrays()

print('Tinh so cap input-output cho tap train/dev (khong materialize vao RAM)...')
train_sample_count = count_caption_samples(train_descriptions, train_features_conv)
dev_sample_count = count_caption_samples(dev_descriptions, dev_features_conv)
print(f'  Train sample count: {train_sample_count:,}')
print(f'  Dev sample count: {dev_sample_count:,}')
print('  Du lieu se duoc stream theo batch tu features_conv (49, 512) de train attention model.')


Tinh so cap input-output cho tap train/dev (khong materialize vao RAM)...
  Train sample count: 306,404
  Dev sample count: 51,622
  Du lieu se duoc stream theo batch tu features_conv (49, 512) de train attention model.


## 7. Xay dung attention model

Cell nay dinh nghia mo hinh attention dung `features_conv` cua VGG16 va word embeddings co the khoi tao bang GloVe.


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    LSTM,
    Embedding,
    Dropout,
    AdditiveAttention,
    Concatenate,
    Flatten,
    Reshape,
    GlobalAveragePooling1D,
)


def define_model_attention(
    vocab_size,
    max_length,
    learning_rate=5e-5,
    embedding_matrix=None,
    embedding_dim=100,
    embedding_trainable=False,
):
    """Định nghĩa image captioning model dùng VGG16 conv features + attention + GloVe tùy chọn."""
    image_input = Input(shape=(49, 512), name='image_regions_input')
    image_regions = Dropout(0.3, name='image_dropout')(image_input)
    image_regions = Dense(256, activation='relu', name='image_projection')(image_regions)
    image_regions = Dropout(0.2, name='image_projection_dropout')(image_regions)
    image_global = GlobalAveragePooling1D(name='image_global_pool')(image_regions)

    sequence_input = Input(shape=(max_length,), name='sequence_input')
    embedding_kwargs = dict(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name='word_embedding')
    if embedding_matrix is not None:
        embedding_kwargs['weights'] = [embedding_matrix]
        embedding_kwargs['trainable'] = embedding_trainable
    sequence_embedding = Embedding(**embedding_kwargs)(sequence_input)
    sequence_embedding = Dropout(0.3, name='embedding_dropout')(sequence_embedding)
    _, state_h, _ = LSTM(
        256,
        return_sequences=True,
        return_state=True,
        dropout=0.2,
        recurrent_dropout=0.0,
        name='caption_lstm',
    )(sequence_embedding)

    query = Reshape((1, 256), name='attention_query')(state_h)
    attention_context = AdditiveAttention(name='image_attention')([query, image_regions])
    attention_context = Flatten(name='flatten_attention_context')(attention_context)

    decoder = Concatenate(name='decoder_concat')([attention_context, image_global, state_h])
    decoder = Dense(256, activation='relu', name='decoder_dense')(decoder)
    decoder = Dropout(0.3, name='decoder_dropout')(decoder)
    outputs = Dense(vocab_size, activation='softmax', name='word_distribution')(decoder)

    optimizer = Adam(learning_rate=learning_rate, clipnorm=1.0)
    model = Model(inputs=[image_input, sequence_input], outputs=outputs, name='caption_attention_model')
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=optimizer,
        metrics=['sparse_categorical_accuracy'],
    )
    return model


tf.keras.backend.clear_session()
model_lstm = define_model_attention(
    vocab_size,
    max_length,
    embedding_matrix=embedding_matrix,
    embedding_dim=embedding_dim,
    embedding_trainable=embedding_trainable,
)
model_lstm.summary()


## 8. Huan luyen mo hinh attention

Phan nay train model captioning su dung `features_conv` cua VGG16 va co co che attention de hoc tot hon bo cuc khong gian cua anh.


## 8.1. Cau hinh train attention

Cell nay cau hinh duong dan model, callback, dataset train/dev, va cac hyperparameters de huan luyen mo hinh attention.


In [ ]:
MODEL_DIR = WORK_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / 'caption_model_attention.keras'
history_path = MODEL_DIR / 'caption_model_attention_history.json'

force_train = True
epochs = 40
batch_size = 64
learning_rate = 5e-5

clear_legacy_training_arrays()
gc.collect()

train_dataset = build_caption_dataset(
    tokenizer, max_length, train_descriptions, train_features_conv, batch_size=batch_size, shuffle=True
).repeat()
dev_dataset = build_caption_dataset(
    tokenizer, max_length, dev_descriptions, dev_features_conv, batch_size=batch_size, shuffle=False
).repeat()

steps_per_epoch = math.ceil(train_sample_count / batch_size)
validation_steps = math.ceil(dev_sample_count / batch_size)

(sample_inputs, sample_targets) = next(iter(train_dataset.take(1)))
sample_images, sample_sequences = sample_inputs

callbacks = [
    ModelCheckpoint(
        filepath=str(model_path),
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1,
    ),
]

print(f'Lưu model tại: {model_path}')
print(f'Lưu history tại: {history_path}')
print(f'force_train: {force_train}')
print(f'Epochs: {epochs}, batch_size: {batch_size}, learning_rate: {learning_rate}')
print(f'Kích thước vocab train attention: {vocab_size}')
print(f'Embedding dim: {embedding_dim}, embedding_trainable: {embedding_trainable}')
print(f'GloVe active: {embedding_matrix is not None}')
print(f'Train samples: {train_sample_count:,}, steps/epoch: {steps_per_epoch}')
print(f'Dev samples: {dev_sample_count:,}, validation_steps: {validation_steps}')
print(f'Sample image batch: {sample_images.shape}, dtype={sample_images.dtype}')
print(f'Sample sequence batch: {sample_sequences.shape}, dtype={sample_sequences.dtype}')
print(f'Sample target batch: {sample_targets.shape}, dtype={sample_targets.dtype}')
print('Dataset train/dev da repeat theo epoch de tranh OUT_OF_RANGE khi train nhieu epoch.')
print('Attention model dang dung features_conv (49, 512) thay vi features_fc (4096).')


## 8.2. Train attention model

Cell nay thuc hien huan luyen hoac load checkpoint attention model tuy theo `force_train` va trang thai file model.


In [ ]:
history = None

if model_path.exists() and not force_train:
    print('Đã tìm thấy model attention đã train trước đó, sẽ load để evaluate/inference.')
    model_lstm = load_model(model_path)
else:
    tf.keras.backend.clear_session()
    model_lstm = define_model_attention(
        vocab_size,
        max_length,
        learning_rate=learning_rate,
        embedding_matrix=embedding_matrix,
        embedding_dim=embedding_dim,
        embedding_trainable=embedding_trainable,
    )
    if model_path.exists() and force_train:
        print('Đã có checkpoint attention cũ, nhưng force_train=True nên sẽ train lại và cập nhật best model.')
    else:
        print('Chưa có model attention đã train, bắt đầu huấn luyện...')
    history = model_lstm.fit(
        train_dataset,
        validation_data=dev_dataset,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        callbacks=callbacks,
        verbose=1,
    )
    with open(history_path, 'w', encoding='utf-8') as history_file:
        json.dump(history.history, history_file, ensure_ascii=False, indent=2)
    print('Huấn luyện xong và đã lưu history.')

if model_path.exists():
    model_lstm = load_model(model_path)
    print('Đã load best attention model từ checkpoint.')


## 8.3. Danh gia validation

Cell nay danh gia nhanh checkpoint tot nhat tren tap validation sau khi train.


In [ ]:
eval_results = model_lstm.evaluate(dev_dataset, steps=validation_steps, verbose=0)
print(f'Validation loss: {eval_results[0]:.4f}')
if len(eval_results) > 1:
    print(f'Validation sparse_categorical_accuracy: {eval_results[1]:.4f}')

if history_path.exists():
    with open(history_path, 'r', encoding='utf-8') as history_file:
        saved_history = json.load(history_file)
    print('Các metric đã lưu:', list(saved_history.keys()))

## 9. Test model tren mot anh

Cell duoi day se:
- load best checkpoint neu can,
- chon 1 anh trong tap test,
- sinh caption bang greedy decoding,
- in caption du doan va caption tham chieu de so sanh.


## 9.1. Test mot anh voi greedy decoding

Cell nay sinh caption cho 1 anh test bang greedy decoding de kiem tra nhanh chat luong mo hinh.


In [ ]:
from PIL import Image
# Ensure matplotlib is installed

%pip install matplotlib
import matplotlib.pyplot as plt


def word_for_id(integer, tokenizer):
    """Tra ve token tu id; ho tro ca vocab bi gioi han boi num_words."""
    if integer <= 0:
        return None
    word = tokenizer.index_word.get(integer)
    if word is None:
        return None
    if tokenizer.num_words is not None and integer >= tokenizer.num_words:
        return None
    return word


def generate_caption_attention(model, tokenizer, photo_feature, max_length):
    """Sinh caption bang greedy decoding tren feature conv cua anh."""
    in_text = 'startseq'
    normalized_feature = normalize_photo_feature(photo_feature)
    if normalized_feature.ndim == 2:
        normalized_feature = normalized_feature.reshape(1, normalized_feature.shape[0], normalized_feature.shape[1])

    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length, padding='post')
        yhat = model.predict([normalized_feature.astype('float32'), sequence.astype('int32')], verbose=0)
        yhat_id = int(np.argmax(yhat[0]))
        word = word_for_id(yhat_id, tokenizer)
        if word is None:
            break
        in_text += ' ' + word
        if word == 'endseq':
            break

    final_tokens = [w for w in in_text.split() if w not in ('startseq', 'endseq')]
    return ' '.join(final_tokens)


def show_test_prediction(sample_image_id=None):
    """Hien thi 1 anh test, caption du doan va cac caption tham chieu."""
    global model_lstm

    if 'model_lstm' not in globals() or model_lstm is None:
        if model_path.exists():
            model_lstm = load_model(model_path)
        else:
            raise FileNotFoundError(f'Khong tim thay model tai {model_path}')

    available_ids = sorted(test_features_conv.keys())
    if not available_ids:
        raise ValueError('Tap test_features_conv dang rong, khong the test model.')

    if sample_image_id is None:
        sample_image_id = available_ids[0]
    elif sample_image_id not in test_features_conv:
        raise KeyError(f'Khong tim thay image_id {sample_image_id} trong tap test.')

    sample_image_path = os.path.join(flickr8k_images, sample_image_id + '.jpg')
    predicted_caption = generate_caption_attention(model_lstm, tokenizer, test_features_conv[sample_image_id], max_length)
    reference_captions = test_descriptions.get(sample_image_id, [])

    print(f'Test image_id: {sample_image_id}')
    print(f'Image path: {sample_image_path}')
    print(f'Predicted caption: {predicted_caption}')
    print('Reference captions:')
    for ref_caption in reference_captions[:5]:
        cleaned_ref = ref_caption.replace('startseq ', '').replace(' endseq', '')
        print(f'  - {cleaned_ref}')

    if os.path.exists(sample_image_path):
        plt.figure(figsize=(8, 6))
        plt.imshow(Image.open(sample_image_path))
        plt.axis('off')
        plt.title(predicted_caption if predicted_caption else 'Predicted caption is empty')
        plt.show()
    else:
        print('Khong mo duoc file anh de hien thi.')

    return sample_image_id, predicted_caption, reference_captions


sample_test_id, sample_prediction, sample_references = show_test_prediction()


## 10. Test model voi beam search

Cell duoi day cai thien buoc sinh caption bang cach:
- dung beam search thay cho greedy decoding,
- chan lap tu lien tiep qua nhieu lan,
- so sanh caption du doan voi caption tham chieu tren anh test.


## 10.1. Test mot anh voi beam search

Cell nay test caption tren 1 anh bang beam search de giam lap tu va cho caption tu nhien hon.


In [ ]:
def generate_caption_beam_search(model, tokenizer, photo_feature, max_length, beam_width=3, repetition_penalty=2):
    """Sinh caption bang beam search va giam lap tu lien tiep tren attention model."""
    normalized_feature = normalize_photo_feature(photo_feature)
    if normalized_feature.ndim == 2:
        normalized_feature = normalized_feature.reshape(1, normalized_feature.shape[0], normalized_feature.shape[1])
    normalized_feature = normalized_feature.astype('float32')

    sequences = [('startseq', 0.0)]

    for _ in range(max_length):
        all_candidates = []
        for seq_text, score in sequences:
            last_tokens = seq_text.split()
            if last_tokens[-1] == 'endseq':
                all_candidates.append((seq_text, score))
                continue

            encoded = tokenizer.texts_to_sequences([seq_text])[0]
            encoded = pad_sequences([encoded], maxlen=max_length, padding='post').astype('int32')
            yhat = model.predict([normalized_feature, encoded], verbose=0)[0]

            candidate_ids = np.argsort(yhat)[-beam_width * 4:][::-1]
            added = 0
            for token_id in candidate_ids:
                word = word_for_id(int(token_id), tokenizer)
                if not word:
                    continue

                penalty = 0.0
                if len(last_tokens) >= repetition_penalty and all(tok == word for tok in last_tokens[-repetition_penalty:]):
                    penalty = 5.0

                prob = float(yhat[token_id])
                candidate_score = score - np.log(prob + 1e-10) + penalty
                all_candidates.append((seq_text + ' ' + word, candidate_score))
                added += 1
                if added >= beam_width:
                    break

        sequences = sorted(all_candidates, key=lambda tup: tup[1])[:beam_width]
        if all(seq.split()[-1] == 'endseq' for seq, _ in sequences):
            break

    best_sequence = sequences[0][0]
    final_tokens = []
    seen_repeats = 0
    prev_word = None
    for word in best_sequence.split():
        if word in ('startseq', 'endseq'):
            continue
        if word == prev_word:
            seen_repeats += 1
            if seen_repeats >= 2:
                continue
        else:
            seen_repeats = 0
        final_tokens.append(word)
        prev_word = word
    return ' '.join(final_tokens)


def show_test_prediction_v2(sample_image_id=None, beam_width=3):
    """Test attention model voi beam search tren 1 anh test."""
    global model_lstm

    if 'model_lstm' not in globals() or model_lstm is None:
        if model_path.exists():
            model_lstm = load_model(model_path)
        else:
            raise FileNotFoundError(f'Khong tim thay model tai {model_path}')

    available_ids = sorted(test_features_conv.keys())
    if not available_ids:
        raise ValueError('Tap test_features_conv dang rong, khong the test model.')

    if sample_image_id is None:
        sample_image_id = available_ids[0]
    elif sample_image_id not in test_features_conv:
        raise KeyError(f'Khong tim thay image_id {sample_image_id} trong tap test.')

    sample_image_path = os.path.join(flickr8k_images, sample_image_id + '.jpg')
    predicted_caption = generate_caption_beam_search(
        model_lstm,
        tokenizer,
        test_features_conv[sample_image_id],
        max_length=max_length,
        beam_width=beam_width,
        repetition_penalty=repetition_penalty,
    )
    reference_captions = test_descriptions.get(sample_image_id, [])

    print(f'Test image_id: {sample_image_id}')
    print(f'Beam width: {beam_width}')
    print(f'Image path: {sample_image_path}')
    print(f'Predicted caption v2: {predicted_caption}')
    print('Reference captions:')
    for ref_caption in reference_captions[:5]:
        cleaned_ref = ref_caption.replace('startseq ', '').replace(' endseq', '')
        print(f'  - {cleaned_ref}')

    if os.path.exists(sample_image_path):
        plt.figure(figsize=(8, 6))
        plt.imshow(Image.open(sample_image_path))
        plt.axis('off')
        plt.title(predicted_caption if predicted_caption else 'Predicted caption is empty')
        plt.show()

    return sample_image_id, predicted_caption, reference_captions


repetition_penalty = 2
sample_test_id_v2, sample_prediction_v2, sample_references_v2 = show_test_prediction_v2(
    sample_image_id='1056338697_4f7d7ce270',
    beam_width=5,
)


## 11. Danh gia tong the bang BLEU score

Cell duoi day tinh BLEU score tren tap test de danh gia tong the chat luong caption:
- BLEU-1: do phu hop unigram,
- BLEU-2: do phu hop den bigram,
- BLEU-3 va BLEU-4: nghiem ngat hon, phan anh chat luong cum tu/cau tot hon.


## 11.1. Tinh BLEU tren tap test

Cell nay tinh BLEU-1 den BLEU-4 tren tap test de danh gia tong the model attention.


In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction


def generate_caption_beam_search_clean(model, tokenizer, photo_feature, max_length, beam_width=5):
    """Wrapper de tao caption test on dinh cho BLEU evaluation."""
    caption = generate_caption_beam_search(
        model=model,
        tokenizer=tokenizer,
        photo_feature=photo_feature,
        max_length=max_length,
        beam_width=beam_width,
        repetition_penalty=repetition_penalty,
    )
    tokens = [token for token in caption.split() if token and token != 'unk']
    return tokens


def evaluate_bleu_scores(model, tokenizer, test_features_conv, test_descriptions, max_length, beam_width=5, limit=None):
    """Tinh BLEU-1..4 tren tap test bang beam search cho attention model."""
    actual, predicted = [], []
    image_ids = sorted(test_features_conv.keys())
    if limit is not None:
        image_ids = image_ids[:limit]

    for idx, image_id in enumerate(image_ids, start=1):
        yhat_tokens = generate_caption_beam_search_clean(
            model=model,
            tokenizer=tokenizer,
            photo_feature=test_features_conv[image_id],
            max_length=max_length,
            beam_width=beam_width,
        )
        ref_captions = []
        for ref in test_descriptions.get(image_id, []):
            tokens = [token for token in ref.split() if token not in ('startseq', 'endseq')]
            ref_captions.append(tokens)

        if ref_captions and yhat_tokens:
            actual.append(ref_captions)
            predicted.append(yhat_tokens)

        if idx % 200 == 0:
            print(f'Da danh gia {idx}/{len(image_ids)} anh test...')

    if not actual:
        raise ValueError('Khong tao duoc du lieu tham chieu/du doan hop le de tinh BLEU.')

    smoothing = SmoothingFunction().method1
    bleu1 = corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0), smoothing_function=smoothing)
    bleu2 = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
    bleu3 = corpus_bleu(actual, predicted, weights=(0.34, 0.33, 0.33, 0), smoothing_function=smoothing)
    bleu4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)

    return {
        'evaluated_images': len(actual),
        'bleu_1': bleu1,
        'bleu_2': bleu2,
        'bleu_3': bleu3,
        'bleu_4': bleu4,
    }


if 'model_lstm' not in globals() or model_lstm is None:
    if model_path.exists():
        model_lstm = load_model(model_path)
    else:
        raise FileNotFoundError(f'Khong tim thay model tai {model_path}')

bleu_limit = None
bleu_results = evaluate_bleu_scores(
    model=model_lstm,
    tokenizer=tokenizer,
    test_features_conv=test_features_conv,
    test_descriptions=test_descriptions,
    max_length=max_length,
    beam_width=5,
    limit=bleu_limit,
)

print(f"So anh duoc danh gia: {bleu_results['evaluated_images']}")
print(f"BLEU-1: {bleu_results['bleu_1']:.4f}")
print(f"BLEU-2: {bleu_results['bleu_2']:.4f}")
print(f"BLEU-3: {bleu_results['bleu_3']:.4f}")
print(f"BLEU-4: {bleu_results['bleu_4']:.4f}")
